# Extreme Gradient Boosting (XGBoost)

**Notebook 2 of 4.** AdaBoost moves weight onto the rows it got wrong and refits.
**Gradient boosting** changes what a tree is fitted to: not reweighted rows, but the error
the trees so far have left behind. `XGBoost` is a gradient boosting library, and section 6
takes up what it puts into the objective on top of that error — a price on the size of each
tree.

In [ ]:
import time
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
from xgboost import XGBRegressor

## 1. The data

`data/california_housing.csv`: 20,640 districts from the 1990 California census — this
notebook calls one row a **district**. Six numbers describing its housing, the two
coordinates, and `location`, a four-value column saying roughly where in the state it sits.

The target `MedHouseVal` is the district's **median house value in units of $100,000**, so
an error of 0.5 is $50,000. That is a number, so this is regression, and error is reported
as **RMSE in those same units**.

`location` is the only column that is not a number, so it is one-hot encoded here exactly as
`01_adaboost.ipynb` encodes the census columns. `03_catboost.ipynb` takes up what to do when
that approach stops working.

In [ ]:
df = pd.read_csv("data/california_housing.csv")

y = df["MedHouseVal"]
# location is the one non-numeric column, and four values one-hot into four columns
X = pd.get_dummies(df.drop(columns="MedHouseVal"), columns=["location"]).astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"{len(X_train)} training districts, {len(X_test)} test districts, {X.shape[1]} columns")
print(f"values run {y.min():.2f} to {y.max():.2f}, in hundreds of thousands, mean {y.mean():.2f}")

## 2. A bar to clear

A random forest is the strong, no-effort baseline: 200 trees grown in parallel on bootstrap
samples, each split choosing from a fraction of the columns.

In [ ]:
forest = RandomForestRegressor(
    # 0.3 scored best here of the fractions tried; the regressor's default of 1.0
    # would give plain bagging instead
    n_estimators=200, max_features=0.3, random_state=42, n_jobs=-1
)
forest.fit(X_train, y_train)

forest_rmse = root_mean_squared_error(y_test, forest.predict(X_test))
print(f"random forest: {forest_rmse:.3f}")

## 3. XGBoost out of the box

Same data, no preparation, no scaling, no tuning.

In [ ]:
start = time.perf_counter()
# no settings at all: 100 trees, learning_rate 0.3, max_depth 6
xgb = XGBRegressor(random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
elapsed = time.perf_counter() - start

train_rmse = root_mean_squared_error(y_train, xgb.predict(X_train))
test_rmse = root_mean_squared_error(y_test, xgb.predict(X_test))

print(f"xgboost: {test_rmse:.3f} in {elapsed:.1f} seconds")
print(f"random forest was {forest_rmse:.3f}")
print(f"\non the training districts it scores {train_rmse:.3f}")

**0.458 against the forest's 0.472**, with nothing tuned, and **0.268 on the training
districts**: the model has very nearly memorized what it was fitted on.

One split is one number, though, and 0.014 is a thin margin to read off a single draw.
Five folds give five numbers, and the spread between them says whether the gap means
anything.

In [ ]:
cv = KFold(5, shuffle=True, random_state=42)


def folds(model, label):
    """Cross-validate a model on the training districts and report mean and spread.

    Args:
        model: any unfitted regressor.
        label: what to call it in the printed line.

    Returns:
        numpy.ndarray: the five fold RMSEs, so two models can be compared fold by fold.
    """
    # negated because scikit-learn scorers are always "higher is better"
    # no n_jobs here: the model inside already uses every core, and asking for the folds
    # in parallel as well would put five of those on top of each other
    scores = -cross_val_score(
        model, X_train, y_train, cv=cv,
        scoring="neg_root_mean_squared_error",
    )
    print(f"{label:<18} mean {scores.mean():.3f}   spread {scores.max() - scores.min():.3f}"
          f"   folds {' '.join(f'{v:.3f}' for v in scores)}")
    return scores


forest_folds = folds(
    RandomForestRegressor(n_estimators=200, max_features=0.3, random_state=42, n_jobs=-1),
    "random forest",
)
xgb_folds = folds(XGBRegressor(random_state=42, n_jobs=-1), "xgboost defaults")

# both models saw the same five folds, so the fold-by-fold result is the comparison to read
print(f"\nxgboost ahead in {(xgb_folds < forest_folds).sum()} of the 5 folds")

**0.474 against 0.479, with a spread of 0.028 between folds.** The distance between the two
models is 0.005; the distance between one fold and the next is five times that, and the
forest takes fold three outright.

So the single split overstated it. Untuned XGBoost **matches** the forest here rather than
beating it, and the 0.268 training score says where the room to improve is. Sections 5 and 6
are where it actually pulls ahead.

## 4. The two settings that work against each other

Boosting adds trees one at a time, each fitted on what is left over. Two settings govern it:

- **`learning_rate`** scales down every tree's contribution. At 0.1 a tree moves the
  prediction a tenth as far as it would like to.
- **`n_estimators`** is how many trees get added.

Smaller steps need more of them, so these two are read **together**. A small rate with too
few trees never arrives; a large rate with many fits noise once it has.

> **Exercise 1.** Fit an `XGBRegressor` for every combination of `learning_rate` in
> `0.3, 0.1, 0.01` and `n_estimators` in `50, 300, 2000`, printing the test RMSE as a 3x3
> table. A minute or two, most of it in the 2,000-tree fits. Which cell is worst, and which
> rate is still improving at 2,000 trees?

<details><summary>
Click here for a hint...
</summary>

`product(rates, counts)` from `itertools` gives the nine pairs in one loop; collect them
in a dictionary keyed by the pair and `pd.DataFrame` lays the result out. Keep
`random_state=42` on every fit.

</details>

In [ ]:
# your code: test RMSE for each learning_rate x n_estimators combination

<details><summary>
Click here to compare your answer...
</summary>

| `learning_rate` | 50 trees | 300 trees | 2000 trees |
| :-- | --: | --: | --: |
| 0.30 | 0.467 | 0.457 | 0.460 |
| 0.10 | 0.491 | 0.442 | **0.438** |
| 0.01 | 0.849 | 0.528 | 0.450 |

**0.01 with 50 trees is the disaster: 0.849.** Predicting the training mean for every
district scores 1.145, so fifty steps at a hundredth of the distance each have barely left
the starting point.

Each row tells the same story: every rate eventually arrives, and the smaller the rate the
longer it takes. Down the last column they have nearly converged, 0.438 to 0.460.

The default 0.3 is already past its best by 2,000 trees, drifting from 0.457 back to 0.460,
while 0.1 is still improving. Finding that out took nine fits, and the next section gets the
tree count from one.

## 5. Letting it find its own stopping point

Picking `n_estimators` by hand means rerunning the search whenever anything else changes.
**Early stopping** does it in one fit: hold out a slice of the training districts, score the
model on it after every tree, and stop once the score has not improved for
`early_stopping_rounds` trees running. The slice comes out of the **training** data.

In [ ]:
# a validation slice carved out of the training districts, never out of the test districts
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

watched = XGBRegressor(
    n_estimators=5000,            # an upper bound, not a plan: stopping decides the real count
    learning_rate=0.05,
    early_stopping_rounds=50,     # 50 trees with no improvement on the slice ends the fit
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1,
)
# eval_set is what gets watched; verbose=False silences one line per tree
watched.fit(X_fit, y_fit, eval_set=[(X_val, y_val)], verbose=False)

print(f"stopped after {watched.best_iteration + 1} trees of the 5000 allowed")
print(f"test RMSE: {root_mean_squared_error(y_test, watched.predict(X_test)):.3f}")

It settles on **992 trees** unprompted and scores **0.444** on the test districts, found in
one fit rather than the nine the grid cost.

It is not free: 3,303 of the 16,512 training districts go to watching rather than fitting.
On a table this size that is a cheap price for the tree count; on a few hundred rows the
slice can cost more than knowing the stopping point is worth.

Section 3 is the reason to check this one on folds too.

In [ ]:
# the stopped model is just these settings, so the folds can score them directly
stopped_folds = folds(
    XGBRegressor(n_estimators=992, learning_rate=0.05, random_state=42, n_jobs=-1),
    "992 trees @ 0.05",
)

print(f"\nbetter than the defaults in {(stopped_folds < xgb_folds).sum()} of the 5 folds")

**0.456 against the defaults' 0.474**, and ahead in **all five folds**, by between 0.010 and
0.023.

This is the comparison section 3 failed. The gap is still smaller than the 0.028 spread
between folds, but that spread is the wrong yardstick here: both models were scored on the
same five folds, so what counts is that one of them won every single one.

## 6. What XGBoost adds: paying for complexity

A plain gradient-boosting tree is grown to fit its errors as closely as it can. XGBoost puts
the **size of the tree into the objective it minimizes**, so every extra leaf has to justify
itself against a penalty. Section 3's 0.268 against 0.458 says the defaults are not charging
enough.

The settings that raise the price:

- **`gamma`** sets the loss reduction a split must buy before it is allowed, so each
  extra leaf is charged for.
- **`max_depth`** caps how deep any tree goes; 3 to 6 covers most uses.
- **`reg_lambda`** penalizes large leaf values, exactly as ridge regression does.
- **`min_child_weight`** refuses splits that would leave too little data in a leaf.
- **`subsample`** fits each tree on a fraction of the rows, drawn fresh each round.
- **`colsample_bytree`** gives each tree a fraction of the columns, as `max_features`
  does in a forest.

The last two are the bagging idea reappearing **inside** boosting.

> **Exercise 2.** Beat **0.458**. Fit at least four `XGBRegressor` configurations from
> the settings above, printing training and test RMSE for each so you can watch the gap
> between them close. A lower `learning_rate` with more trees gives the penalties something
> to work with, so vary those too.

<details><summary>
Click here for a hint...
</summary>

One helper taking keyword arguments and returning both RMSEs makes each configuration a
single line. `subsample` and `colsample_bytree` around 0.8 are the usual starting point.

</details>

In [ ]:
# your code: several configurations, each reporting training and test RMSE

<details><summary>
Click here to compare your answer...
</summary>

| configuration | train | test |
| :-- | --: | --: |
| defaults | 0.268 | 0.458 |
| `max_depth=3` | 0.448 | 0.492 |
| `reg_lambda=50` | 0.336 | **0.448** |
| depth 4, rate 0.05, 500 trees, `subsample=0.8`, `colsample_bytree=0.8` | 0.403 | 0.468 |

Watch the training column rise from 0.268 to 0.336 as the test column falls from 0.458 to
0.448. **Getting worse on the training data is the point**; a tuning run where training error
keeps dropping is not working.

The ridge penalty alone wins. The other two charge too much: `max_depth=3` and the slow,
sampled combination push training error past 0.4 and give back more than they buy, landing at
0.492 and 0.468, both **worse than the untouched defaults**. A penalty is a price, and it can
be overpaid.

**0.448**, against 0.458 untuned and the forest's 0.472 — and section 5 reached 0.444 without
a search at all.

One caveat on that number. The configurations were compared **on the test set**, so 0.448 is
an optimistic estimate of what this model would do on new districts: part of the margin is
the search finding the test set's own quirks. Cross-validating on the training districts is
how to choose settings and keep the test score meaningful.

</details>

## Summary

- Gradient boosting fits each tree on **what is left over**.
- Untuned XGBoost **matched** the random forest rather than beating it: 0.474 against 0.479
  cross-validated, where a single split had said 0.458 against 0.472.
- **`learning_rate` and `n_estimators` are one setting in two parts**: rate 0.01 scored
  0.849 with 50 trees and 0.450 with 2,000, while 0.3 had drifted past its best by then.
- **Early stopping** found 992 trees in a single fit, and those settings beat the defaults
  in **all five folds**, 0.456 against 0.474.
- XGBoost **charges for tree complexity**, so tuning means watching training error get
  *worse*: 0.268 to 0.336, as test fell 0.458 to 0.448. Two of the four configurations
  charged too much and finished behind the defaults.

Next, in `03_catboost.ipynb`: the columns one-hot encoding handles badly.